# **Data Preprocessing**

In [ ]:
import pandas as pd
import re

from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [ ]:
dataset = load_dataset(
    "salehalmansour/english-to-arabic-translate"
)

df = dataset["train"].to_pandas()

print("Columns in dataset:", df.columns.tolist())
print("Dataset shape:", df.shape)

df.head()

Repo card metadata block was not found. Setting CardData to empty.


Columns in dataset: ['en', 'ar']
Dataset shape: (1325899, 2)


,en,ar
0,and this,و هذه؟
1,it was um,...لقد كان
2,what is she doing here,ما الذي تفعله هناك؟
3,i dont like it,لا أحب ذلك
4,did you get the part,هل حصلت على جزء ?


In [ ]:
df = df.dropna(subset=["en", "ar"])

print("After removing nulls:", df.shape)

After removing nulls: (1325887, 2)


In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(r'[^\w\s]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def normalize_arabic(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)

    return text

In [ ]:
df["en"] = df["en"].apply(clean_text)

df["ar"] = df["ar"].apply(
    lambda x: normalize_arabic(clean_text(x))
)

df.head()

,en,ar
0,and this,و هذه
1,it was um,لقد كان
2,what is she doing here,ما الذي تفعله هناك
3,i dont like it,لا احب ذلك
4,did you get the part,هل حصلت علي جزء


In [ ]:
df = df.sample(
    n=200000,
    random_state=42
)

print("Sampled dataset size:", df.shape)

Sampled dataset size: (200000, 2)


In [ ]:
train, temp = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val, test = train_test_split(
    temp,
    test_size=0.5,
    random_state=42
)

print("Train size:", len(train))
print("Validation size:", len(val))
print("Test size:", len(test))

Train size: 160000
Validation size: 20000
Test size: 20000


In [ ]:
import os

os.makedirs(
    "data/processed",
    exist_ok=True
)

train.to_csv(
    "data/processed/train.csv",
    index=False
)

val.to_csv(
    "data/processed/val.csv",
    index=False
)

test.to_csv(
    "data/processed/test.csv",
    index=False
)

print("Dataset processed successfully!")

Dataset processed successfully!


In [ ]:
print("Final dataset summary:")
print("Total sampled:", len(df))
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Final dataset summary:
Total sampled: 200000
Train: 160000
Validation: 20000
Test: 20000


# **Tokenization**

In [ ]:
import pandas as pd
import torch
import os
import json
import pickle
from collections import Counter
from typing import List, Tuple, Dict, Optional

In [ ]:
# Configuration
MAX_SEQ_LEN = 50
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 15000

# Special tokens
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

print(" Imports loaded")
print(f"Special tokens: {SPECIAL_TOKENS}")

 Imports loaded
Special tokens: ['<pad>', '<sos>', '<eos>', '<unk>']


In [ ]:
class TranslationTokenizer:
    def __init__(self, max_len: int = MAX_SEQ_LEN):
        self.max_len = max_len
        self.PAD_TOKEN = PAD_TOKEN
        self.SOS_TOKEN = SOS_TOKEN
        self.EOS_TOKEN = EOS_TOKEN
        self.UNK_TOKEN = UNK_TOKEN
        self.special_tokens = SPECIAL_TOKENS

        self.ar_stoi = None
        self.ar_itos = None
        self.en_stoi = None
        self.en_itos = None

    def get_special_id(self, token_type: str) -> int:
        """Get ID for special token (pad, sos, eos, unk)"""
        mapping = {
            'pad': self.PAD_TOKEN,
            'sos': self.SOS_TOKEN,
            'eos': self.EOS_TOKEN,
            'unk': self.UNK_TOKEN
        }
        return self.ar_stoi[mapping[token_type]]

    def tokenize(self, sentence: str, lang: str) -> List[str]:
        """Simple whitespace tokenization"""
        return sentence.split()

    def build_vocabulary(self, arabic_sentences: List[str], english_sentences: List[str],
                         min_freq: int = MIN_WORD_FREQ, max_size: int = MAX_VOCAB_SIZE):
        """Build vocabularies from training data"""

        # Count frequencies
        ar_counter = Counter()
        for sent in arabic_sentences:
            ar_counter.update(sent.split())

        en_counter = Counter()
        for sent in english_sentences:
            en_counter.update(sent.split())

        # Build vocab with special tokens
        ar_vocab = self.special_tokens.copy()
        en_vocab = self.special_tokens.copy()

        # Add Arabic words by frequency
        for word, freq in ar_counter.most_common(max_size):
            if freq >= min_freq:
                ar_vocab.append(word)

        # Add English words by frequency
        for word, freq in en_counter.most_common(max_size):
            if freq >= min_freq:
                en_vocab.append(word)

        # Create mappings
        self.ar_stoi = {word: idx for idx, word in enumerate(ar_vocab)}
        self.ar_itos = ar_vocab
        self.en_stoi = {word: idx for idx, word in enumerate(en_vocab)}
        self.en_itos = en_vocab

        print(f" Arabic vocab size: {len(self.ar_stoi)}")
        print(f" English vocab size: {len(self.en_stoi)}")

    def numericalize(self, tokens: List[str], lang: str) -> List[int]:
        """Convert tokens to indices"""
        stoi = self.ar_stoi if lang == 'ar' else self.en_stoi
        unk_id = stoi[self.UNK_TOKEN]
        return [stoi.get(token, unk_id) for token in tokens]

    def add_special_tokens(self, indices: List[int], add_sos: bool = True,
                          add_eos: bool = True) -> List[int]:
        """Add SOS and EOS tokens"""
        result = []
        if add_sos:
            result.append(self.get_special_id('sos'))
        result.extend(indices)
        if add_eos:
            result.append(self.get_special_id('eos'))
        return result

    def pad_sequence(self, sequence: List[int]) -> List[int]:
        """Pad or truncate sequence to max_len"""
        pad_id = self.get_special_id('pad')

        # Truncate if too long
        if len(sequence) > self.max_len:
            # Keep SOS and EOS if they exist at the ends after truncation
            if sequence[0] == self.get_special_id('sos') and sequence[-1] == self.get_special_id('eos'):
                sequence = sequence[:self.max_len - 1] + [sequence[-1]]
            else:
                sequence = sequence[:self.max_len]

        # Pad if too short
        if len(sequence) < self.max_len:
            sequence = sequence + [pad_id] * (self.max_len - len(sequence))

        return sequence

    def encode_sentence(self, sentence: str, lang: str, add_sos: bool = True,
                       add_eos: bool = True, pad: bool = True) -> torch.Tensor:
        """Full encoding pipeline"""
        tokens = self.tokenize(sentence, lang)
        indices = self.numericalize(tokens, lang)
        if add_sos or add_eos:
            indices = self.add_special_tokens(indices, add_sos, add_eos)
        if pad:
            indices = self.pad_sequence(indices)
        return torch.tensor(indices, dtype=torch.long)

    def decode(self, indices: List[int], lang: str, skip_special: bool = True) -> str:
        """Convert indices back to sentence"""
        itos = self.ar_itos if lang == 'ar' else self.en_itos
        tokens = []
        for idx in indices:
            token = itos[idx]
            if skip_special and token in self.special_tokens:
                continue
            tokens.append(token)
        return ' '.join(tokens)

    def process_dataframe(self, df: pd.DataFrame, lang: str) -> torch.Tensor:
        """Process entire dataframe column"""
        column = 'ar' if lang == 'ar' else 'en'
        sentences = df[column].tolist()

        encoded = []
        for sent in sentences:
            # Explicitly cast each sentence to string to avoid AttributeError with floats
            encoded_tensor = self.encode_sentence(str(sent), lang)
            encoded.append(encoded_tensor)

        return torch.stack(encoded)

    def save(self, save_dir: str):
        """Save tokenizer to disk"""
        os.makedirs(save_dir, exist_ok=True)

        tokenizer_state = {
            'max_len': self.max_len,
            'ar_stoi': self.ar_stoi,
            'ar_itos': self.ar_itos,
            'en_stoi': self.en_stoi,
            'en_itos': self.en_itos,
            'special_tokens': self.special_tokens
        }

        with open(os.path.join(save_dir, 'tokenizer.pkl'), 'wb') as f:
            pickle.dump(tokenizer_state, f)

        # Save as JSON for inspection
        with open(os.path.join(save_dir, 'ar_vocab.json'), 'w', encoding='utf-8') as f:
            json.dump({'stoi': self.ar_stoi, 'itos': self.ar_itos},
                     f, ensure_ascii=False, indent=2)

        with open(os.path.join(save_dir, 'en_vocab.json'), 'w', encoding='utf-8') as f:
            json.dump({'stoi': self.en_stoi, 'itos': self.en_itos},
                     f, ensure_ascii=False, indent=2)

        print(f" Tokenizer saved to {save_dir}")

    @classmethod
    def load(cls, load_dir: str):
        """Load tokenizer from disk"""
        with open(os.path.join(load_dir, 'tokenizer.pkl'), 'rb') as f:
            tokenizer_state = pickle.load(f)

        tokenizer = cls(max_len=tokenizer_state['max_len'])
        tokenizer.ar_stoi = tokenizer_state['ar_stoi']
        tokenizer.ar_itos = tokenizer_state['ar_itos']
        tokenizer.en_stoi = tokenizer_state['en_stoi']
        tokenizer.en_itos = tokenizer_state['en_itos']
        tokenizer.special_tokens = tokenizer_state['special_tokens']

        print(f" Tokenizer loaded from {load_dir}")
        return tokenizer

print(" Tokenizer class defined successfully!")

 Tokenizer class defined successfully!


In [ ]:
# Set paths
base_dir = os.getcwd()
processed_dir = os.path.join(base_dir, 'data', 'processed')

# Load data
print(" Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(processed_dir, 'train.csv'))
val_df = pd.read_csv(os.path.join(processed_dir, 'val.csv'))
test_df = pd.read_csv(os.path.join(processed_dir, 'test.csv'))

print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

train_df.head()

 Loading preprocessed data...
Train: 160000 samples
Val: 20000 samples
Test: 20000 samples


,en,ar
0,this provision will be offset by a correspondi...,1 تقني
1,organizational matter 3637,دال المسائل التنظيميه 3940
2,the country programme 20052006,البرنامج القطري 20052006
3,date of loss,الجدول 19 تواريخ الخساره
4,ii requirements for filing claims,2 اشتراطات لتقديم المطالبات


In [ ]:
import pandas as pd
import re
from collections import Counter
import os

# Initialize tokenizer
print(" Initializing tokenizer...")
tokenizer = TranslationTokenizer(max_len=MAX_SEQ_LEN)

# Build vocabulary from training data only
print("\n Building vocabulary from training data...")

# Ensure all sentences are strings before passing them to the tokenizer
arabic_sentences = [str(s) for s in train_df['ar'].tolist()]
english_sentences = [str(s) for s in train_df['en'].tolist()]

tokenizer.build_vocabulary(
    arabic_sentences=arabic_sentences,
    english_sentences=english_sentences,
    min_freq=MIN_WORD_FREQ,
    max_size=MAX_VOCAB_SIZE
)

# Show vocab stats
print(f"\n Vocabulary Statistics:")
print(f"   Arabic - Unique words: {len(tokenizer.ar_stoi)}")
print(f"   English - Unique words: {len(tokenizer.en_stoi)}")

 Initializing tokenizer...

 Building vocabulary from training data...
 Arabic vocab size: 15004
 English vocab size: 15004

 Vocabulary Statistics:
   Arabic - Unique words: 15004
   English - Unique words: 15004


In [ ]:
# Process all splits
print(" Processing datasets...")

train_ar = tokenizer.process_dataframe(train_df, 'ar')
train_en = tokenizer.process_dataframe(train_df, 'en')

val_ar = tokenizer.process_dataframe(val_df, 'ar')
val_en = tokenizer.process_dataframe(val_df, 'en')

test_ar = tokenizer.process_dataframe(test_df, 'ar')
test_en = tokenizer.process_dataframe(test_df, 'en')

print(f"\n Tensor shapes:")
print(f"   Train: {train_ar.shape}, {train_en.shape}")
print(f"   Val: {val_ar.shape}, {val_en.shape}")
print(f"   Test: {test_ar.shape}, {test_en.shape}")

 Processing datasets...

 Tensor shapes:
   Train: torch.Size([160000, 50]), torch.Size([160000, 50])
   Val: torch.Size([20000, 50]), torch.Size([20000, 50])
   Test: torch.Size([20000, 50]), torch.Size([20000, 50])


In [ ]:
# Save tensors
tensor_dir = os.path.join(base_dir, '..', 'data', 'tensors')
os.makedirs(tensor_dir, exist_ok=True)

torch.save(train_ar, os.path.join(tensor_dir, 'train_ar.pt'))
torch.save(train_en, os.path.join(tensor_dir, 'train_en.pt'))
torch.save(val_ar, os.path.join(tensor_dir, 'val_ar.pt'))
torch.save(val_en, os.path.join(tensor_dir, 'val_en.pt'))
torch.save(test_ar, os.path.join(tensor_dir, 'test_ar.pt'))
torch.save(test_en, os.path.join(tensor_dir, 'test_en.pt'))

# Save tokenizer
tokenizer_dir = os.path.join(base_dir, '..', 'models', 'tokenizer')
tokenizer.save(tokenizer_dir)

print(" All files saved successfully!")

 Tokenizer saved to /content/../models/tokenizer
 All files saved successfully!


In [ ]:
# Test encoding/decoding
sample_idx = 0

original_ar = train_df.iloc[sample_idx]['ar']
original_en = train_df.iloc[sample_idx]['en']

encoded_ar = train_ar[sample_idx].tolist()
encoded_en = train_en[sample_idx].tolist()

decoded_ar = tokenizer.decode(encoded_ar, 'ar')
decoded_en = tokenizer.decode(encoded_en, 'en')

print(" Sample Test:")
print(f"\nOriginal Arabic:  {original_ar}")
print(f"Decoded Arabic:   {decoded_ar}")
print(f"\nOriginal English: {original_en}")
print(f"Decoded English:  {decoded_en}")

print("\nTokenizer working perfectly!")

 Sample Test:

Original Arabic:  1 تقني
Decoded Arabic:   1 تقني

Original English: this provision will be offset by a corresponding increase under income section 1 income from staff assessment
Decoded English:  this provision will be offset by a corresponding increase under income section 1 income from staff assessment

Tokenizer working perfectly!


# **Sequance Processing**

In [ ]:

from torch.utils.data import Dataset,DataLoader

Max_Vec_Len=50
Batch_size=32

#Already padded in tokenizer
encoder_input = train_en
decoder_sequences = train_ar

val_encoder_input =val_ar.clone()
val_decoder_sequences =val_en.clone()

In [ ]:
#Decoder
decoder_input=decoder_sequences[:,:-1]
decoder_target=decoder_sequences[:,1:]

val_decoder_input=val_decoder_sequences[:,:-1]
val_decoder_target=val_decoder_sequences[:,1:]

In [ ]:
#Convert to long Type(already tensors)

encoder_input=encoder_input.long()
decoder_input=decoder_input.long()
decoder_target=decoder_target.long()


val_encoder_input=val_encoder_input.long()
val_decoder_input=val_decoder_input.long()
val_decoder_target=val_decoder_target.long()



In [ ]:
class TranslationDataset(Dataset):
      def __init__(self,enc,dec_in,dec_out):
          self.enc=enc
          self.dec_in=dec_in
          self.dec_out =dec_out
      def __len__(self):
        return len(self.enc)

      def __getitem__(self,idx):
        return{
            "encoder_input":self.enc[idx],
            "decoder_input":self.dec_in[idx],
            "decoder_target":self.dec_out[idx]

        }

In [ ]:
#DataLoader

train_dataset=TranslationDataset(encoder_input,decoder_input,decoder_target)
val_dataset=TranslationDataset(val_encoder_input,val_decoder_input,val_decoder_target)

train_loader=DataLoader(
    train_dataset,
    batch_size=Batch_size,
    shuffle=True  #randomize
)

val_loader=DataLoader(
    val_dataset,
    batch_size=Batch_size,
    shuffle=False  #don't
)

In [ ]:
#Test

batch=next(iter(train_loader))

print("Encoder Input Shape:",batch["encoder_input"].shape)
print("Decoder Input Shape:",batch["decoder_input"].shape)
print("Decoder Target Shape:",batch["decoder_target"].shape)

Encoder Input Shape: torch.Size([32, 50])
Decoder Input Shape: torch.Size([32, 49])
Decoder Target Shape: torch.Size([32, 49])


# **Architect Model (Tranformer)**

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]

In [ ]:
class TransformerModel(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=2048,
        dropout=0.1,
        max_len=50
    ):
        super().__init__()

        self.d_model = d_model

        # Embedding layers
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len)

        # Transformer core
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        # Final output layer
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        self.dropout = nn.Dropout(dropout)

    def generate_square_subsequent_mask(self, sz):
        # Prevent decoder from seeing future tokens
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

    def forward(self, src, tgt):
        """
        src: [batch, src_len]
        tgt: [batch, tgt_len]
        """

        # Embedding + scaling
        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)

        # Add positional encoding
        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)

        # Create target mask (for decoder)
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)

        # Transformer forward
        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask
        )

        # Final linear layer
        output = self.fc_out(output)

        return output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerModel(
    src_vocab_size=len(tokenizer.en_stoi),  # English
    tgt_vocab_size=len(tokenizer.ar_stoi),  # Arabic
).to(device)

In [ ]:
batch = next(iter(train_loader))

src = batch["encoder_input"].to(device)   # English
tgt = batch["decoder_input"].to(device)   # Arabic (shifted)

output = model(src, tgt)

print(output.shape)

torch.Size([32, 49, 15004])
